In [0]:
CATALOG = 'SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'ORDERS_ENRICHED'

In [0]:
orders_df = spark.table('SALES.BRONZE.ORDERS')
orders_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

orders_enriched_df = orders_df.select(
trim(col("order_id")).alias("order_id"),

trim(col("customer_id")).alias("customer_id"),

upper(trim(col("order_status"))).alias("order_status"),

#T1

date_format(col("order_purchase_timestamp"), 'yyyy-MM-dd').alias("purchase_date"), 

date_format(col("order_approved_at"), 'yyyy-MM-dd').alias("approved_date"), 

date_format(col("order_delivered_carrier_date"), 'yyyy-MM-dd').alias("delivered_date"), 

to_timestamp(col("order_delivered_customer_date")).alias("order_delivered_customer_date"),

date_format(col("order_estimated_delivery_date"), 'yyyy-MM-dd').alias("estimated_date"),

#T2

hour(col("order_purchase_timestamp")).alias("purchase_hour"), 

dayofweek(col("order_purchase_timestamp")).alias("purchase_weekday"), 

date_format(col("order_purchase_timestamp"),"E").alias("purchase_weekday_name"), 

when(col("order_delivered_customer_date").isNotNull(), hour(col("order_delivered_customer_date")) ).alias("delivery_hour"), 

when(col("order_delivered_customer_date").isNotNull(), dayofweek(col("order_delivered_customer_date")) ).alias("delivered_weekday"), 

when(col("order_delivered_customer_date").isNotNull(), date_format(col("order_delivered_customer_date"),"E") ).alias("delivered_weekday_name"),

#T3

datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")).alias("delivery_days"), 

when(col("order_delivered_customer_date").isNotNull(), datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")) ).alias("delay_days"),

#T4

when(col("purchase_hour").between(6, 11), 'MORNING') .when(col("purchase_hour").between(12, 17), 'AFTERNOON') .when(col("purchase_hour").between(18, 22), 'EVENING') .otherwise('NIGHT') .alias("purchase_time_of_day"),

#T5

when(col("order_status").isin('Delivered', 'Invoice'), 'COMPLETED') .when(col("order_status").isin('Cancelled', 'Unvailable'), 'CANCELLED') .otherwise('PENDING').alias("order_status_group"),

#T6

### when(col("order_approved_at") < col("order_purchase_timestamp")) | (col("order_delivered_carrier_date") < col("order_approved_at")) | (col("order_delivered_customer_date") < col("order_delivered_carrier_date"),
###                                                                                                                                       

col("file_name").alias("file_name"),

col("file_path").alias("file_path"),

col("load_timestamp").alias("load_timestamp")

).filter('order_id IS NOT NULL')

display(orders_enriched_df.limit(1))

In [0]:
orders_enriched_df.createOrReplaceTempView('enriched_orders_temp_vw')

In [0]:
spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_orders_temp_vw
  )
  WHERE rn = 1
) src
ON (src.order_id  = tgt.order_id)
WHEN MATCHED THEN
  UPDATE SET
    tgt.order_id = src.order_id,
    tgt.customer_id = src.customer_id,
    tgt.order_status = src.order_status,
    tgt.purchase_date = src.purchase_date,
    tgt.approved_date = src.approved_date,
    tgt.delivered_date = src.delivered_date,
    tgt.order_delivered_customer_date = src.order_delivered_customer_date,
    tgt.estimated_date = src.estimated_date,
    tgt.purchase_hour = src.purchase_hour,
    tgt.purchase_weekday = src.purchase_weekday,
    tgt.purchase_weekday_name = src.purchase_weekday_name,
    tgt.delivery_hour = src.delivery_hour,
    tgt.delivered_weekday = src.delivered_weekday,
    tgt.delivered_weekday_name = src.delivered_weekday_name,
    tgt.delivery_days = src.delivery_days,
    tgt.delay_days = src.delay_days,
    tgt.purchase_time_of_day = src.purchase_time_of_day,
    tgt.order_status_group = src.order_status_group


WHEN NOT MATCHED THEN
  INSERT (
    order_id,
    customer_id,
    order_status,
    purchase_date,
    approved_date,
    delivered_date,
    order_delivered_customer_date,
    estimated_date,
    purchase_hour,
    purchase_weekday,
    purchase_weekday_name,
    delivery_hour,
    delivered_weekday,
    delivered_weekday_name,
    delivery_days,
    delay_days,
    purchase_time_of_day,
    order_status_group,
    file_name,
    origin_file_path,
    load_timestamp
  )
  VALUES 
  (
    src.order_id,
    src.customer_id,
    src.order_status,
    src.purchase_date,
    src.approved_date,
    src.delivered_date,
    src.order_delivered_customer_date,
    src.estimated_date,
    src.purchase_hour,
    src.purchase_weekday,
    src.purchase_weekday_name,
    src.delivery_hour,
    src.delivered_weekday,
    src.delivered_weekday_name,
    src.delivery_days,
    src.delay_days,
    src.purchase_time_of_day,
    src.order_status_group,
    src.file_name,
    src.file_path,
    current_timestamp()
  )
""")
